# Ch 29. 데이터 파이프라인 — PII · 합성 · IAA

운영 데이터의 세 관문: PII 마스킹 4단계, LLM 합성 라벨, Inter-Annotator Agreement(Cohen's κ), 데이터 버전 관리.

In [ ]:
# !pip install -q scikit-learn transformers

import re
import hashlib
import json
import math
import random
from pathlib import Path

print("환경 준비 완료")

## 최소 예제 — PII 마스킹 (정규식)

4단계 중 **1단계**: 정규식으로 전화번호·카드번호·주민번호·이메일·계좌번호를 잡는다.

정규식만으로 약 90% 의 PII 를 잡을 수 있다.

In [ ]:
PII_PATTERNS = {
    "PHONE":   r"01[016789][\-\s]?\d{3,4}[\-\s]?\d{4}",
    "CARD":    r"\d{4}[\-\s]?\d{4}[\-\s]?\d{4}[\-\s]?\d{4}",
    "RRN":     r"\d{6}[\-\s]?[1-4]\d{6}",                 # 주민번호
    "EMAIL":   r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}",
    "ACCOUNT": r"\d{2,4}-\d{2,4}-\d{4,8}",                # 계좌
}

def mask_regex(text):
    for tag, pat in PII_PATTERNS.items():
        text = re.sub(pat, f"[{tag}]", text)
    return text

# 테스트
sample = "고객 번호 010-1234-5678, 이메일 user@example.com, 계좌 123-456-12345678"
print("원본:  ", sample)
print("마스킹:", mask_regex(sample))

## 최소 예제 — IAA (Cohen's κ)

두 라벨러(사람 또는 LLM)의 합치도를 정량화.

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

- **κ ≥ 0.6** 이 실용 임계. 미달이면 라벨 정의 재작성 후 재측정.

In [ ]:
from sklearn.metrics import cohen_kappa_score

# 라벨러 A 와 B 가 같은 100건에 라벨 (시뮬레이션)
random.seed(42)
labels_a = [random.choice(["pos", "neg", "neu"]) for _ in range(100)]

# B는 A와 약 80% 일치하는 라벨러
labels_b = [
    a if random.random() < 0.8 else random.choice(["pos", "neg", "neu"])
    for a in labels_a
]

k = cohen_kappa_score(labels_a, labels_b)
print(f"κ = {k:.2f}")
# κ < 0.6 → 라벨 정의 모호. 가이드라인 재작성.
if k < 0.6:
    print("→ κ < 0.6: 라벨 정의 모호. 가이드라인 재작성 필요.")
else:
    print("→ κ ≥ 0.6: 양호. 본격 라벨링 진행 가능.")

## 실전 — PII 마스킹 4단계 파이프라인

정규식(1단계) → NER(2단계) → LLM 검증(3단계) → 사람 검수(4단계) 통합.

In [ ]:
# 단계 2: NER 모델 (실제 운영에서는 transformers pipeline 사용)
# from transformers import pipeline
# ner = pipeline("token-classification", model="ner_pii_model",
#                aggregation_strategy="simple")
# def mask_ner(text):
#     for ent in ner(text):
#         text = text.replace(ent["word"], f"[{ent['entity_group']}]")
#     return text

def mask_ner(text):
    """NER 마스킹 플레이스홀더 (이름·주소 대상)."""
    # 실제로는 Ch 25 NER 모델 호출
    return text

def full_pii_pipeline(text):
    """4단계 PII 마스킹 파이프라인."""
    # 단계 1: 정규식 (~90%)
    text = mask_regex(text)
    # 단계 2: NER — 이름·주소·기관명 (+5%)
    text = mask_ner(text)
    # 단계 3: LLM 검증 — 잔여 PII (+3~4%) [운영에서는 API 호출]
    # text = llm_verify(text)
    # 단계 4: 사람 검수 100~500건 샘플 (+0.5~1%)
    return text

test_cases = [
    "홍길동 고객님 전화번호 010-9876-5432 로 연락 바랍니다.",
    "이메일 admin@company.co.kr 로 영수증 발송 완료.",
    "카드번호 1234 5678 9012 3456 으로 결제 완료.",
    "주민번호 901201-1234567 확인 요청.",
]

for tc in test_cases:
    masked = full_pii_pipeline(tc)
    print(f"원본:   {tc}")
    print(f"마스킹: {masked}")
    print()

## 실전 — 데이터 버전 관리 (hash + 메타파일)

학습 모델 → 사용 데이터 → 합성 시점 → PII 마스킹 버전. 사슬 추적 필수.

In [ ]:
def data_hash(jsonl_path):
    """파일 내용 hash."""
    h = hashlib.md5()
    with open(jsonl_path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()[:12]

# 실습용 더미 JSONL 파일 생성
dummy_path = Path("/tmp/domain_pairs.jsonl")
dummy_path.write_text(
    "\n".join([json.dumps({"input": f"입력{i}", "output": f"출력{i}"}, ensure_ascii=False) for i in range(100)])
)

# 메타파일 저장
meta = {
    "data_path":        str(dummy_path),
    "data_hash":        data_hash(dummy_path),
    "synthesized_at":   "2026-04-26",
    "teacher_model":    "claude-haiku-4-5",
    "filter_version":   "v3 (k>=3, len>=150, no-meta)",
    "pii_mask_version": "v2 (regex+NER+LLM)",
    "iaa_kappa":        0.78,
    "size":             48732,
    "license":          "internal",
}
meta_path = Path("/tmp/data_meta.json")
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False))

print("생성된 메타파일:")
print(meta_path.read_text())

## 연습문제

1. 본인 도메인의 raw 100 문장에 §2 의 4단계 PII 마스킹 적용. 잡힌 PII 비율 측정.
2. 같은 100 문장을 라벨러 A (본인) 와 라벨러 B (Haiku LLM) 가 라벨. κ 측정.
3. κ < 0.6 인 케이스 발견 시 — 어디서 합의 부족? 라벨 정의 재작성.
4. 합성 5K 페어의 메타파일 작성 (hash + 정의 + IAA).
5. **(생각해볼 것)** "PII 마스킹 99%" 가 안전한가? 1% 의 누락이 회수되면 어떤 결과?

In [ ]:
# 연습 1: 본인 도메인 100 문장에 PII 마스킹 적용 + 잡힌 비율 측정


In [ ]:
# 연습 2: 라벨러 A vs B (또는 Haiku LLM) Cohen's κ 측정


In [ ]:
# 연습 3: κ < 0.6 케이스 분석 + 라벨 정의 재작성


In [ ]:
# 연습 4: 합성 5K 페어 메타파일 작성
